# 07: Empirical Validation（经验验证）

目标：用真实微博数据对照验证模型的核心经验假设（H1-H4），并在两套标注数据上做同方法复核：

- **核心对照集**：`outputs/annotations/master/long_covid_annotations_master.jsonl`（17,604）+ `dataset/Topic_data/merged_topic_official.csv`
- **扩展集（Batch3）**：`outputs/annotations/batches/batch_03_expanded/new_batch3.jsonl`（73,456）+ `outputs/annotations/intermediate/to_annotate_batch3_clean.csv`

统一口径：
- 暂不做“按话题分组比较”（后续再补 topic mapping）。
- 先做同一套指标与统计检验，在 **核心对照集 / 扩展集 / 合并样本** 上分别报告结果，检查稳健性。

## H1-H4 与研究问题的对应
- **H1（Activity→Jump）**：$a$ 越高（中立者越少），系统越容易出现突变式变化（用 $|d|Q|/dt|$ 峰值/突变指标衡量）。
- **H2（r_proxy→Volatility）**：$r\_{proxy}$ 越高（自媒体相对更占优），波动性越大（用 $\mathrm{std}(Q)$ / rolling volatility）。
- **H3（r×a 交互）**：高 $r\_{proxy}$ 且高 $a$ 的窗口/分段应更“脆弱”（波动/突变更大）。
- **H4（临界慢化）**：突变前应出现 AC1↑、Var↑ 的早期预警信号（经验数据里可能被外生冲击与噪声掩盖，需客观报告）。


In [ ]:
# ===== 0. Imports & Paths =====
from __future__ import annotations

import json
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

# 设定项目根目录
ROOT = Path("..").resolve() if Path("..").resolve().name == "emotion_dynamics" else Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.empirical import load_topic_dataset, aggregate_time_series, UserTypeMapper
from src.empirical.time_series import TimeSeriesConfig, calculate_r_proxy, calculate_rolling_ac1, calculate_rolling_stats

# 输出目录
FIG_DIR = ROOT / "outputs/figs/empirical"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# 中文字体（若缺失会自动回退）
plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

print("ROOT:", ROOT)
print("FIG_DIR:", FIG_DIR)


In [ ]:
# ===== 1. 读取与对齐：dataset csv + annotations jsonl =====

@dataclass(frozen=True)
class DatasetSpec:
    name: str
    dataset_csv: Path
    annotations_jsonl: Path


def load_annotations_jsonl(path: Path) -> pd.DataFrame:
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            records.append(obj)
    if not records:
        raise ValueError(f"标注文件为空：{path}")
    df = pd.DataFrame(records)
    if "mid" not in df.columns:
        raise ValueError(f"标注文件缺少 mid：{path}")
    df["mid"] = df["mid"].astype(str)
    keep = [c for c in ["mid", "emotion_class", "risk_class", "emotion_confidence", "risk_confidence"] if c in df.columns]
    df = df[keep].drop_duplicates(subset=["mid"]).reset_index(drop=True)
    return df


def load_and_merge(spec: DatasetSpec, *, mapper: Optional[UserTypeMapper] = None) -> pd.DataFrame:
    mapper = mapper or UserTypeMapper()
    df_raw = load_topic_dataset(spec.dataset_csv, mapper=mapper)
    if "mid" not in df_raw.columns:
        raise ValueError(f"dataset 缺少 mid 列：{spec.dataset_csv}")
    df_raw["mid"] = df_raw["mid"].astype(str)
    df_raw = df_raw.drop_duplicates(subset=["mid"]).reset_index(drop=True)

    df_ann = load_annotations_jsonl(spec.annotations_jsonl)

    df = df_raw.merge(
        df_ann,
        on=["mid"],
        how="inner",
        validate="one_to_one",
    )
    coverage = len(df) / max(len(df_raw), 1)
    print(f"[{spec.name}] raw={len(df_raw):,} ann={len(df_ann):,} merged={len(df):,} coverage={coverage:.2%}")
    return df


MASTER = DatasetSpec(
    name="master",
    dataset_csv=ROOT / "dataset/Topic_data/merged_topic_official.csv",
    annotations_jsonl=ROOT / "outputs/annotations/master/long_covid_annotations_master.jsonl",
)

BATCH3 = DatasetSpec(
    name="batch3",
    dataset_csv=ROOT / "outputs/annotations/intermediate/to_annotate_batch3_clean.csv",
    annotations_jsonl=ROOT / "outputs/annotations/batches/batch_03_expanded/new_batch3.jsonl",
)

for s in [MASTER, BATCH3]:
    if not s.dataset_csv.exists():
        raise FileNotFoundError(s.dataset_csv)
    if not s.annotations_jsonl.exists():
        raise FileNotFoundError(s.annotations_jsonl)

print("OK: 数据文件存在")


In [ ]:
# ===== 2. 加载两套数据，并构造合并样本 =====

mapper = UserTypeMapper()

df_master = load_and_merge(MASTER, mapper=mapper)
df_batch3 = load_and_merge(BATCH3, mapper=mapper)

df_all = pd.concat([df_master, df_batch3], ignore_index=True)
df_all = df_all.drop_duplicates(subset=["mid"]).reset_index(drop=True)

print("[all] total merged:", f"{len(df_all):,}")
print("time span (master):", df_master["publish_time"].min(), "~", df_master["publish_time"].max())
print("time span (batch3):", df_batch3["publish_time"].min(), "~", df_batch3["publish_time"].max())
print("time span (all):", df_all["publish_time"].min(), "~", df_all["publish_time"].max())

display(df_all[["user_type"]].value_counts().rename("count").head(10))


In [ ]:
# ===== 3. 聚合为时间序列（统一口径） =====

FREQ = "1H"          # 可改："4H" / "1D"（稳健性分析可补）
MIN_POSTS_PUBLIC = 5  # 公众帖子太少的窗口会将 X_H/X_M/X_L/a/Q 置为 NaN

def build_time_series(df: pd.DataFrame, *, freq: str, min_posts: int) -> pd.DataFrame:
    cfg = TimeSeriesConfig(freq=freq, min_posts=int(min_posts))
    ts = aggregate_time_series(df, config=cfg)
    ts["r_proxy"] = calculate_r_proxy(ts)
    return ts

ts_master = build_time_series(df_master, freq=FREQ, min_posts=MIN_POSTS_PUBLIC)
ts_batch3 = build_time_series(df_batch3, freq=FREQ, min_posts=MIN_POSTS_PUBLIC)
ts_all = build_time_series(df_all, freq=FREQ, min_posts=MIN_POSTS_PUBLIC)

print("valid windows (master):", int(ts_master["a"].notna().sum()), "/", len(ts_master))
print("valid windows (batch3):", int(ts_batch3["a"].notna().sum()), "/", len(ts_batch3))
print("valid windows (all):", int(ts_all["a"].notna().sum()), "/", len(ts_all))

# 可选：落盘，方便后续复用（不会覆盖旧文件，统一加后缀）
out_dir = ROOT / "outputs/annotations/derived"
out_dir.mkdir(parents=True, exist_ok=True)
ts_master.to_csv(out_dir / f"time_series_master_{FREQ.lower()}.csv", index=False)
ts_batch3.to_csv(out_dir / f"time_series_batch3_{FREQ.lower()}.csv", index=False)
ts_all.to_csv(out_dir / f"time_series_all_{FREQ.lower()}.csv", index=False)
print("saved:", out_dir)


In [ ]:
# ===== 4. 快速可视化：Q / a / r_proxy =====

def plot_basic(ts: pd.DataFrame, title: str, *, out_name: str) -> None:
    df = ts.copy()
    df = df.sort_values("time_window").reset_index(drop=True)
    fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
    axes[0].plot(df["time_window"], df["Q"], lw=1)
    axes[0].set_ylabel("Q")
    axes[0].set_title(title)
    axes[1].plot(df["time_window"], df["a"], lw=1)
    axes[1].set_ylabel("a")
    axes[2].plot(df["time_window"], df["r_proxy"], lw=1)
    axes[2].set_ylabel("r_proxy")
    axes[2].set_xlabel("time")
    for ax in axes:
        ax.grid(True, alpha=0.2)
    fig.tight_layout()
    fig.savefig(FIG_DIR / out_name, dpi=200)
    plt.show()

plot_basic(ts_master, f"Master: Q/a/r_proxy ({FREQ})", out_name=f"fig7a_master_basic_{FREQ.lower()}.png")
plot_basic(ts_batch3, f"Batch3: Q/a/r_proxy ({FREQ})", out_name=f"fig7a_batch3_basic_{FREQ.lower()}.png")
plot_basic(ts_all, f"All (master+batch3): Q/a/r_proxy ({FREQ})", out_name=f"fig7a_all_basic_{FREQ.lower()}.png")


In [ ]:
# ===== 5. 指标构造：jump / volatility / rolling AC1+Var =====

def add_window_metrics(ts: pd.DataFrame, *, vol_win: int = 12) -> pd.DataFrame:
    """为窗口级别构造 jump/volatility 指标。

    经验数据的时间窗可能存在缺口（并非严格连续 1H 序列），因此 jump 使用“按实际时间差归一化”的形式：
        abs_dQ_abs_per_hour = |Δ|Q|| / Δt_hours
    以避免因为缺口导致的“伪大跳跃”。
    """
    df = ts.copy().sort_values("time_window").reset_index(drop=True)
    df["Q_abs"] = df["Q"].abs()
    df["dt_hours"] = df["time_window"].diff().dt.total_seconds() / 3600.0
    df.loc[df["dt_hours"] <= 0, "dt_hours"] = np.nan
    df["dQ_abs_per_hour"] = df["Q_abs"].diff() / df["dt_hours"]
    df["abs_dQ_abs_per_hour"] = df["dQ_abs_per_hour"].abs()
    df["Q_volatility"] = df["Q"].rolling(vol_win, min_periods=max(3, vol_win // 3)).std()
    return df


def segment_metrics(df: pd.DataFrame, *, segment: str = "M") -> pd.DataFrame:
    """把时间序列分段（默认按月），用段内统计量检验 H1-H3。

    为降低小样本窗口带来的噪声：
    - a_mean 默认按 n_public 做加权平均；
    - r_proxy_mean 采用“段内媒体计数比值”（sum 计数）而非窗口级 ratio 的简单平均。
    """
    x = df.dropna(subset=["time_window"]).copy()
    x["seg"] = x["time_window"].dt.to_period(segment).dt.to_timestamp()
    rows = []
    for seg, g in x.groupby("seg"):
        g_valid = g.dropna(subset=["a", "Q", "abs_dQ_abs_per_hour"])
        if len(g_valid) < 10:
            continue

        # a：按公众帖子数加权更稳健
        if "n_public" in g_valid.columns:
            w = g_valid["n_public"].fillna(0).astype(float).values
            a_mean = float(np.average(g_valid["a"].values, weights=w)) if float(w.sum()) > 0 else float(g_valid["a"].mean())
        else:
            a_mean = float(g_valid["a"].mean())

        # r_proxy：段内媒体计数比值（避免窗口级 ratio 噪声）
        if "n_mainstream" in g_valid.columns and "n_wemedia" in g_valid.columns:
            nw = float(g_valid["n_wemedia"].fillna(0).sum())
            nm = float(g_valid["n_mainstream"].fillna(0).sum())
            r_proxy_mean = float(nw / (nw + nm)) if (nw + nm) > 0 else np.nan
        else:
            r_proxy_mean = float(g_valid["r_proxy"].mean()) if "r_proxy" in g_valid.columns else np.nan
        rows.append(
            {
                "seg": seg,
                "n_windows": len(g_valid),
                "a_mean": a_mean,
                "r_proxy_mean": r_proxy_mean,
                "volatility": float(g_valid["Q"].std()),
                "jump_score": float(g_valid["abs_dQ_abs_per_hour"].max()),
            }
        )
    return pd.DataFrame(rows).sort_values("seg").reset_index(drop=True)


def safe_pearsonr(x: pd.Series, y: pd.Series):
    try:
        from scipy import stats
        m = x.notna() & y.notna()
        if int(m.sum()) < 5:
            return np.nan, np.nan
        r, p = stats.pearsonr(x[m].values, y[m].values)
        return float(r), float(p)
    except Exception:
        return np.nan, np.nan


def run_h1_h2_h3(ts: pd.DataFrame, *, name: str) -> pd.DataFrame:
    df = add_window_metrics(ts)
    seg = segment_metrics(df, segment="M")
    if seg.empty:
        print(f"[{name}] 段内有效样本不足，无法检验 H1-H3")
        return seg

    r1, p1 = safe_pearsonr(seg["a_mean"], seg["jump_score"])  # H1
    r2, p2 = safe_pearsonr(seg["r_proxy_mean"], seg["volatility"])  # H2

    print(f"\n[{name}] segments={len(seg)}")
    print(f"H1: corr(a_mean, jump_score) = {r1:.3f} (p={p1:.4f})")
    print(f"H2: corr(r_proxy_mean, volatility) = {r2:.3f} (p={p2:.4f})")

    # H3: 简单分组对照 + 回归（可选 statsmodels）
    a_med = float(seg["a_mean"].median())
    r_med = float(seg["r_proxy_mean"].median())
    hi = seg[(seg["a_mean"] > a_med) & (seg["r_proxy_mean"] > r_med)]
    lo = seg[(seg["a_mean"] <= a_med) & (seg["r_proxy_mean"] <= r_med)]
    print(f"H3(group): high-high n={len(hi)}, mean(vol)={hi['volatility'].mean():.4f}; low-low n={len(lo)}, mean(vol)={lo['volatility'].mean():.4f}")

    try:
        import statsmodels.formula.api as smf
        m = seg.dropna(subset=["volatility", "a_mean", "r_proxy_mean"]).copy()
        if len(m) >= 10:
            model = smf.ols("volatility ~ a_mean * r_proxy_mean", data=m).fit()
            print("H3(reg): volatility ~ a_mean * r_proxy_mean")
            print(model.summary().tables[1])
    except Exception:
        print("H3(reg): statsmodels 不可用，跳过回归（不影响主结论）。")

    # 图：H1/H2 散点
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].scatter(seg["a_mean"], seg["jump_score"], s=25, alpha=0.8)
    axes[0].set_xlabel("a_mean (segment)")
    axes[0].set_ylabel("jump_score (max |d|Q|/dt|)")
    axes[0].set_title(f"H1 ({name})")
    axes[0].grid(True, alpha=0.2)

    axes[1].scatter(seg["r_proxy_mean"], seg["volatility"], s=25, alpha=0.8)
    axes[1].set_xlabel("r_proxy_mean (segment)")
    axes[1].set_ylabel("volatility (std(Q))")
    axes[1].set_title(f"H2 ({name})")
    axes[1].grid(True, alpha=0.2)

    fig.tight_layout()
    fig.savefig(FIG_DIR / f"fig7b_h1_h2_scatter_{name}_{FREQ.lower()}.png", dpi=200)
    plt.show()

    return seg


seg_master = run_h1_h2_h3(ts_master, name="master")
seg_batch3 = run_h1_h2_h3(ts_batch3, name="batch3")
seg_all = run_h1_h2_h3(ts_all, name="all")


In [ ]:
# ===== 6. H4：临界慢化（事件对齐的 AC1/Var） =====

def pick_jump_events(df: pd.DataFrame, *, q_col: str = "abs_dQ_abs_per_hour", quantile: float = 0.95, min_gap: int = 8):
    x = df.dropna(subset=["time_window", q_col]).copy().sort_values("time_window").reset_index(drop=True)
    if len(x) < 50:
        return []
    thr = float(x[q_col].quantile(quantile))
    cand = x.index[x[q_col] >= thr].tolist()
    events = []
    last = -10**9
    for idx in cand:
        if idx - last < int(min_gap):
            continue
        events.append(idx)
        last = idx
    return events


def event_study(df: pd.DataFrame, event_idx: list[int], *, col: str, pre: int = 24):
    mats = []
    for idx in event_idx:
        if idx - pre < 0:
            continue
        w = df.loc[idx - pre : idx, col].values
        if np.isnan(w).any():
            continue
        mats.append(w)
    if not mats:
        return None
    mat = np.vstack(mats)
    mean = np.mean(mat, axis=0)
    lo = np.percentile(mat, 2.5, axis=0)
    hi = np.percentile(mat, 97.5, axis=0)
    return mean, lo, hi, mat


def run_h4(ts: pd.DataFrame, *, name: str, roll_win: int = 12, pre: int = 24):
    df = add_window_metrics(ts)
    df = calculate_rolling_ac1(df, column="Q", window_size=int(roll_win))
    df = calculate_rolling_stats(df, window_size=int(roll_win), columns=["Q"])  # 生成 Q_rolling_var

    events = pick_jump_events(df, q_col="abs_dQ_abs_per_hour", quantile=0.95, min_gap=max(6, roll_win // 2))
    print(f"[{name}] jump events picked: {len(events)}")

    ac_col = "Q_rolling_ac1"
    var_col = "Q_rolling_var"

    ac = event_study(df, events, col=ac_col, pre=pre)
    var = event_study(df, events, col=var_col, pre=pre)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    xs = np.arange(-pre, 1)

    if ac is not None:
        mean, lo, hi, _ = ac
        axes[0].plot(xs, mean, lw=2)
        axes[0].fill_between(xs, lo, hi, alpha=0.2)
    axes[0].axvline(0, color="gray", linestyle="--", lw=1)
    axes[0].set_title(f"H4: AC1 before jumps ({name})")
    axes[0].set_xlabel("windows to jump")
    axes[0].set_ylabel("AC1")
    axes[0].grid(True, alpha=0.2)

    if var is not None:
        mean, lo, hi, _ = var
        axes[1].plot(xs, mean, lw=2)
        axes[1].fill_between(xs, lo, hi, alpha=0.2)
    axes[1].axvline(0, color="gray", linestyle="--", lw=1)
    axes[1].set_title(f"H4: Var before jumps ({name})")
    axes[1].set_xlabel("windows to jump")
    axes[1].set_ylabel("Var")
    axes[1].grid(True, alpha=0.2)

    fig.tight_layout()
    fig.savefig(FIG_DIR / f"fig7c_h4_eventstudy_{name}_{FREQ.lower()}.png", dpi=200)
    plt.show()

run_h4(ts_master, name="master", roll_win=12, pre=24)
run_h4(ts_batch3, name="batch3", roll_win=12, pre=24)
run_h4(ts_all, name="all", roll_win=12, pre=24)


## 结论整理（写作建议）

建议在论文中按“可复现口径”写：
- 先报告三套口径（master / batch3 / all）的 H1-H4 方向是否一致；
- 再解释若 H4 不稳定/不显著：经验数据存在外生冲击、平台机制与观测噪声，可能淹没临界慢化信号（这并不否认理论机制，而是提示识别困难与需要更精细的因果设计）。

下一步（后续补充）：
- 做 topic mapping 后按话题对照；
- 对窗口长度（1H/4H/1D）、min_posts 阈值、jump 定义阈值做稳健性扫描。
